# FaIR climate model simulations — extended scenarios

Runs the [FaIR v2.2](https://github.com/OMS-NetZero/FAIR) climate model
over 1750–2501 across seven ScenarioMIP-aligned scenarios (VL through HL)
and saves the ensemble outputs to NetCDF for downstream plotting in
`0505_extensions_plotting.ipynb`.

This notebook is **simulation only** — no plots are produced here.
The figures in the paper are rendered in 0505 from the saved outputs.

**Steps**
1. Define the time horizon, scenarios, and species.
2. Load the AR6-calibrated parameter ensemble (Zenodo).
3. Fill emissions + volcanic/solar forcing from CSV.
4. Compute total GHG emissions in CO₂-equivalent (AR6 GWP100).
5. Run FaIR.
6. Save outputs to `data/fair-outputs/fair_run.nc`.


## Imports

In [1]:
import os

import numpy as np
import pandas as pd
import pooch
import xarray as xr
from fair import FAIR
from fair.interface import initialise
from fair.io import read_properties


/Users/bensan/ScenarioMIP_final/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Set `memory_limited = True` to run only 5 ensemble members (for quick
testing). The full AR6 ensemble (~1500 members) is fetched from Zenodo
on first run.


In [2]:
memory_limited = False


## Define scenarios, time horizon, and species

In [3]:
snames = ["VL", "LN", "L", "ML", "M", "H", "HL"]

f = FAIR()
f.define_time(1750, 2501, 1)
f.define_scenarios(snames)

species, properties = read_properties("../data/fair-inputs/species_configs_properties_1.4.1.csv")
f.define_species(species, properties)
f.ch4_method = "Thornhill2021"


## Load the calibrated parameter ensemble

In [4]:
if not memory_limited:
    ZENODO_DOI = "10.5281/zenodo.7112539"
    FILE_NAME = "calibrated_constrained_parameters.csv"
    FILE_HASH = "md5:8a70a3fb05d0e0cf35e136de382582a5"

    data_pooch = pooch.create(
        path="../data/fair-inputs",
        base_url=f"doi:{ZENODO_DOI}",
        version="1.5.0",
        registry={FILE_NAME: FILE_HASH},
    )
    local_file_path = data_pooch.fetch(FILE_NAME)
    print(f"Config file: {local_file_path}")


Config file: /Users/bensan/ScenarioMIP_final/data/fair-inputs/1.5.0/calibrated_constrained_parameters.csv


In [5]:
if memory_limited:
    df_configs = pd.read_csv("../data/fair-inputs/1.5.0/calibrated_constrained_parameters_short.csv", index_col=0)
else:
    df_configs = pd.read_csv("../data/fair-inputs/1.5.0/calibrated_constrained_parameters.csv", index_col=0)

f.define_configs(df_configs.index)
f.allocate()
print(f"Ensemble size: {len(df_configs)} configs")


Ensemble size: 841 configs


## Fill emissions and natural forcing

In [6]:
f.fill_from_csv(
    forcing_file="../data/fair-inputs/volcanic_solar.csv",
    emissions_file="../data/fair-inputs/emissions_1750-2500.csv",
)

# Solar forcing is handled separately by FaIR (zeroed in the input file).
for s in f.scenarios:
    f.forcing.loc[dict(scenario=s, specie="Solar")] = 0


## CO₂-equivalent emissions

Aggregate all GHG emissions to CO₂-equivalents using 100-year Global
Warming Potentials (AR6 GWP100, mass-adjusted).


In [7]:
gwpmat = pd.read_csv("../data/fair-inputs/gwp_mass_adjusted_100y.csv", index_col=0)

co2eo = f.emissions.sel(specie="CO2 FFI")[:, :, 0].copy() * 0
for specie in f.emissions.specie.values:
    gwp = gwpmat["ar6_gwp_mass_adjusted"].get(specie, np.nan)
    if not np.isnan(gwp):
        co2eo = co2eo + f.emissions.sel(specie=specie)[:, :, 0] * gwp / 1e6
co2e = co2eo * 1e6  # tonnes CO2e per year


## Run FaIR

In [8]:
f.fill_species_configs("../data/fair-inputs/species_configs_properties_1.4.1.csv")
if memory_limited:
    f.override_defaults("../data/fair-inputs/1.5.0/calibrated_constrained_parameters_short.csv")
else:
    f.override_defaults("../data/fair-inputs/1.5.0/calibrated_constrained_parameters.csv")

initialise(f.concentration, f.species_configs["baseline_concentration"])
initialise(f.forcing, 0)
initialise(f.temperature, 0)
initialise(f.cumulative_emissions, 0)
initialise(f.airborne_emissions, 0)
initialise(f.ocean_heat_content_change, 0)

f.run()


Running 5887 projections in parallel:   0%|          | 0/751 [00:00<?, ?timesteps/s]

Running 5887 projections in parallel:   0%|          | 1/751 [00:00<04:03,  3.08timesteps/s]

Running 5887 projections in parallel:   0%|          | 3/751 [00:00<01:41,  7.37timesteps/s]

Running 5887 projections in parallel:   1%|          | 5/751 [00:00<01:22,  8.99timesteps/s]

Running 5887 projections in parallel:   1%|          | 7/751 [00:00<01:12, 10.22timesteps/s]

...

Running 5887 projections in parallel: 100%|█████████▉| 748/751 [03:25<00:01,  2.78timesteps/s]

Running 5887 projections in parallel: 100%|█████████▉| 749/751 [03:26<00:00,  3.05timesteps/s]

Running 5887 projections in parallel: 100%|█████████▉| 750/751 [03:26<00:00,  2.83timesteps/s]

Running 5887 projections in parallel: 100%|██████████| 751/751 [03:26<00:00,  3.12timesteps/s]

Running 5887 projections in parallel: 100%|██████████| 751/751 [03:26<00:00,  3.63timesteps/s]

## Save outputs

Bundle the variables needed by `0505_extensions_plotting.ipynb` into a
single NetCDF file. Emissions are sliced at `config=0` (they are
config-invariant); climate variables retain the full ensemble.


In [9]:
out = xr.Dataset(
    {
        "emissions": f.emissions.isel(config=0, drop=True),
        "co2e": co2e,
        "temperature": f.temperature.isel(layer=0, drop=True),
        "co2_concentration": f.concentration.sel(specie="CO2", drop=True),
        "forcing_sum": f.forcing_sum.rename("forcing_sum"),
    }
)

encoding = {v: {"dtype": "float32", "zlib": True, "complevel": 4} for v in out.data_vars}

out_dir = "../data/fair-outputs"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "fair_run.nc")
out.to_netcdf(out_path, encoding=encoding)

print(f"Saved {out_path}")
print(out)


Saved ../data/fair-outputs/fair_run.nc
<xarray.Dataset> Size: 109MB
Dimensions:            (timepoints: 751, scenario: 7, specie: 61, config: 841,
                        timebounds: 752)
Coordinates:
  * timepoints         (timepoints) float64 6kB 1.75e+03 1.752e+03 ... 2.5e+03
  * scenario           (scenario) <U2 56B 'VL' 'LN' 'L' 'ML' 'M' 'H' 'HL'
  * specie             (specie) <U43 10kB 'CO2 FFI' ... 'Equivalent effective...
  * config             (config) int64 7kB 1299 3919 4204 ... 1593049 1597582
  * timebounds         (timebounds) float64 6kB 1.75e+03 1.751e+03 ... 2.501e+03
Data variables:
    emissions          (timepoints, scenario, specie) float64 3MB 0.008735 .....
    co2e               (timepoints, scenario) float64 42kB 1.472e+06 ... 3.58...
    temperature        (timebounds, scenario, config) float64 35MB 0.0 ... 1.661
    co2_concentration  (timebounds, scenario, config) float64 35MB 277.3 ... ...
    forcing_sum        (timebounds, scenario, config) float64 35MB 